# 47: implementing core data structures yourself

python already gives us excellent built-in data structures - `list`, `dict`, `set`, `collections.deque` - and in
real code you should almost always just use those, they're written in C and extremely well optimized.

so why bother implementing a linked list, a binary search tree or a hash map by hand at all?

* it's the single best way to actually *understand* how the built-ins work under the hood (why is `dict` lookup
  O(1) but `list.index()` is O(n)? why is prepending to a `list` slow but prepending to a `deque` is fast?)
* it's one of the most common things asked in technical coding interviews
* sometimes you genuinely need a specialized variant the standard library doesn't offer

this lesson builds a singly linked list, a stack and a queue, a binary search tree, and a simplified hash map -
each one contrasted directly against the real built-in that already does the same job.

## a singly linked list

a linked list is a chain of `Node` objects, each one holding a value and a reference to the *next* node - there's
no single contiguous block of memory like a python `list` uses internally. that's exactly why some operations
have different costs: prepending to a linked list is O(1) (just point a new head at the old head), while
prepending to a python `list` is O(n) (every existing element has to shift over by one).

In [1]:
class Node:
    # __slots__ keeps each node small - we know exactly which attributes a node will ever have, so we don't
    # need python's normal per-instance __dict__ overhead. a nice small habit for lots-of-tiny-objects classes.
    __slots__ = ("value", "next")

    def __init__(self, value, next=None):
        self.value = value
        self.next = next


class LinkedList:
    def __init__(self):
        self.head = None
        self._length = 0

    def __len__(self):
        return self._length

    def prepend(self, value):
        # O(1): just make a new node point at the current head, then move the head pointer to it
        self.head = Node(value, self.head)
        self._length += 1

    def append(self, value):
        # O(n): we have to walk all the way to the end first, since we don't keep a separate tail pointer here
        new_node = Node(value)
        if self.head is None:
            self.head = new_node
        else:
            current = self.head
            while current.next is not None:
                current = current.next
            current.next = new_node
        self._length += 1

    def find(self, value):
        # O(n): has to walk node by node, there's no way to "jump" to a position like list indexing can
        current = self.head
        while current is not None:
            if current.value == value:
                return True
            current = current.next
        return False

    def delete(self, value):
        # O(n): we need the PREVIOUS node to re-point its .next around the one we're removing
        previous, current = None, self.head
        while current is not None:
            if current.value == value:
                if previous is None:
                    self.head = current.next  # deleting the head itself
                else:
                    previous.next = current.next
                self._length -= 1
                return True
            previous, current = current, current.next
        return False

    def __iter__(self):
        current = self.head
        while current is not None:
            yield current.value
            current = current.next

    def __repr__(self):
        return " -> ".join(str(v) for v in self) or "(empty)"

In [2]:
ll = LinkedList()
ll.append("b")
ll.append("c")
ll.prepend("a")  # O(1) - notice this doesn't need to touch b or c at all
print("list:", ll)
print("length:", len(ll))
print("contains 'c'?", ll.find("c"))
ll.delete("b")
print("after deleting 'b':", ll)
print("as a real list via iteration:", list(ll))

list: a -> b -> c
length: 3
contains 'c'? True
after deleting 'b': a -> c
as a real list via iteration: ['a', 'c']


## stack and queue

a **stack** is last-in-first-out (LIFO) - think a stack of plates, you can only take the top one off. a
**queue** is first-in-first-out (FIFO) - think a line of people, whoever got there first leaves first.

python's plain `list` makes a perfectly good stack (`.append()`/`.pop()` are both O(1) at the *end*), but it
makes a bad queue: `list.pop(0)` is O(n) because everything after index 0 has to shift left. `collections.deque`
is a proper doubly-linked-list-like structure under the hood, so *both* ends are O(1) - that's exactly why it's
the standard library's actual answer to "I need a fast queue" (we first saw `deque` back in lesson 12).

In [3]:
from collections import deque
import time

class Stack:
    def __init__(self):
        self._items = []

    def push(self, item):
        self._items.append(item)  # O(1)

    def pop(self):
        return self._items.pop()  # O(1) - removes and returns the LAST item

    def peek(self):
        return self._items[-1]

    def is_empty(self):
        return len(self._items) == 0


class Queue:
    def __init__(self):
        self._items = deque()

    def enqueue(self, item):
        self._items.append(item)  # O(1) at the right end

    def dequeue(self):
        return self._items.popleft()  # O(1) at the left end - this is the whole reason we use deque, not list

    def is_empty(self):
        return len(self._items) == 0


s = Stack()
for x in ("first", "second", "third"):
    s.push(x)
print("stack pops in LIFO order:", [s.pop() for _ in range(3)])

q = Queue()
for x in ("first", "second", "third"):
    q.enqueue(x)
print("queue dequeues in FIFO order:", [q.dequeue() for _ in range(3)])

# proving list.pop(0) really is slower than deque.popleft() for a queue-shaped workload
N = 20_000
slow_list = list(range(N))
start = time.perf_counter()
while slow_list:
    slow_list.pop(0)
list_duration = time.perf_counter() - start

fast_deque = deque(range(N))
start = time.perf_counter()
while fast_deque:
    fast_deque.popleft()
deque_duration = time.perf_counter() - start

print(f"\ndraining {N} items via list.pop(0): {list_duration:.4f}s")
print(f"draining {N} items via deque.popleft(): {deque_duration:.4f}s")
print(f"deque was roughly {list_duration / deque_duration:.0f}x faster for this pattern")

stack pops in LIFO order: ['third', 'second', 'first']
queue dequeues in FIFO order: ['first', 'second', 'third']

draining 20000 items via list.pop(0): 0.0388s
draining 20000 items via deque.popleft(): 0.0010s
deque was roughly 37x faster for this pattern


## a binary search tree (BST)

a binary search tree keeps every value in a specific order: for any node, everything in its *left* subtree is
smaller, and everything in its *right* subtree is bigger. that ordering is exactly what makes search fast - at
every step we can throw away roughly half the remaining tree, giving O(log n) search on a *balanced* tree
(a plain python `list` needs a full linear O(n) scan to find an arbitrary value, unless it's already sorted and
you use `bisect` - a topic for another day).

In [4]:
class BSTNode:
    __slots__ = ("value", "left", "right")

    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None


class BinarySearchTree:
    def __init__(self):
        self.root = None

    def insert(self, value):
        if self.root is None:
            self.root = BSTNode(value)
            return
        current = self.root
        while True:
            if value < current.value:
                if current.left is None:
                    current.left = BSTNode(value)
                    return
                current = current.left
            else:
                if current.right is None:
                    current.right = BSTNode(value)
                    return
                current = current.right

    def contains(self, value):
        current = self.root
        while current is not None:
            if value == current.value:
                return True
            current = current.left if value < current.value else current.right
        return False

    def in_order(self):
        # yields every value in sorted order - visit left subtree, then this node, then right subtree
        def walk(node):
            if node is None:
                return
            yield from walk(node.left)
            yield node.value
            yield from walk(node.right)
        yield from walk(self.root)

In [5]:
bst = BinarySearchTree()
for value in (50, 30, 70, 20, 40, 60, 80):
    bst.insert(value)

print("is 40 in the tree?", bst.contains(40))
print("is 99 in the tree?", bst.contains(99))
print("every value, in sorted order (in-order traversal):", list(bst.in_order()))

is 40 in the tree? True
is 99 in the tree? False
every value, in sorted order (in-order traversal): [20, 30, 40, 50, 60, 70, 80]


## a simplified hash map

python's `dict` is itself a hash map, and this is the closest we'll get to peeking at how it actually works
inside: values are stored in numbered "buckets", and which bucket a given key lands in is decided by
`hash(key) % number_of_buckets`. that's what makes `dict` lookup O(1) on average - instead of checking every
key one by one, we jump *straight* to the right bucket.

two different keys CAN land in the same bucket though (a "collision") - our implementation handles that with
**chaining**: each bucket just holds a small list of `(key, value)` pairs instead of a single one.

In [6]:
class SimpleHashMap:
    def __init__(self, bucket_count=8):
        self._buckets = [[] for _ in range(bucket_count)]
        self._count = 0

    def _bucket_for(self, key):
        return self._buckets[hash(key) % len(self._buckets)]

    def __setitem__(self, key, value):
        bucket = self._bucket_for(key)
        for i, (existing_key, _) in enumerate(bucket):
            if existing_key == key:
                bucket[i] = (key, value)  # key already exists - overwrite its value
                return
        bucket.append((key, value))
        self._count += 1
        # once buckets get too full on average, growing (and re-hashing everything) keeps lookups fast -
        # this is exactly what CPython's real dict does internally too, just with more tuning
        if self._count > len(self._buckets) * 0.75:
            self._grow()

    def __getitem__(self, key):
        for existing_key, value in self._bucket_for(key):
            if existing_key == key:
                return value
        raise KeyError(key)

    def __contains__(self, key):
        return any(existing_key == key for existing_key, _ in self._bucket_for(key))

    def __len__(self):
        return self._count

    def _grow(self):
        old_items = [pair for bucket in self._buckets for pair in bucket]
        self._buckets = [[] for _ in range(len(self._buckets) * 2)]
        self._count = 0
        for key, value in old_items:  # re-insert everything - each key may land in a totally different bucket now
            self[key] = value

In [7]:
hm = SimpleHashMap(bucket_count=4)
hm["name"] = "Ada"
hm["job"] = "programmer"
hm["name"] = "Ada Lovelace"  # overwrites, doesn't add a second entry
print("name:", hm["name"])
print("'job' in map?", "job" in hm)
print("'missing' in map?", "missing" in hm)
print("total entries:", len(hm))

print("\nbucket layout right now (some keys share a bucket - a collision, handled via chaining):")
for i, bucket in enumerate(hm._buckets):
    print(f"  bucket {i}: {bucket}")

# push it past the 75%-full threshold and watch it grow (re-hash) automatically
for i in range(10):
    hm[f"key{i}"] = i
print(f"\nafter adding 10 more entries, bucket_count grew from 4 to: {len(hm._buckets)}")

name: Ada Lovelace
'job' in map? True
'missing' in map? False
total entries: 2

bucket layout right now (some keys share a bucket - a collision, handled via chaining):
  bucket 0: [('name', 'Ada Lovelace')]
  bucket 1: []
  bucket 2: [('job', 'programmer')]
  bucket 3: []

after adding 10 more entries, bucket_count grew from 4 to: 16


## big-O cheat sheet: our version vs. the real built-in

| structure          | our version                | real built-in            | typical cost                     |
|---------------------|-----------------------------|----------------------------|-----------------------------------|
| singly linked list  | `LinkedList`                | (no direct equivalent)     | prepend O(1), append/find O(n)   |
| stack                | `Stack`                     | plain `list`               | push/pop O(1) at the end          |
| queue                | `Queue`                     | `collections.deque`        | enqueue/dequeue O(1) both ends   |
| binary search tree   | `BinarySearchTree`          | (no direct equivalent)      | search/insert O(log n) *if balanced*, O(n) worst case |
| hash map             | `SimpleHashMap`             | `dict`                      | get/set/contains O(1) on average |

note the "if balanced" caveat on our BST: inserting already-sorted data (1, 2, 3, 4, 5...) degenerates it into
basically a linked list, O(n) search. real production tree structures (like the ones backing database indexes
in lessons 41/42) use *self-balancing* variants (AVL trees, red-black trees, B-trees) specifically to guarantee
O(log n) no matter what order the data arrives in - implementing self-balancing is a great follow-up exercise
once you're comfortable with the plain version above.

## quick comparison to other languages

* **Java**: `LinkedList`, `Stack`/`Deque`, `TreeMap` (a self-balancing red-black tree!) and `HashMap` are all
  right there in the standard library, and are implemented conceptually just like what we built here
* **C#**: `LinkedList<T>`, `Stack<T>`/`Queue<T>`, `SortedDictionary<TKey,TValue>` (also a red-black tree) and
  `Dictionary<TKey,TValue>` - same idea, .NET's collections namespace
* **C++**: `std::forward_list`, `std::stack`/`std::deque`, `std::map` (a red-black tree) and
  `std::unordered_map` (a hash map) - C++ is explicit about which one is tree-ordered vs hash-ordered right in
  the name
* **Go**: no built-in linked list/tree/hashmap types as such, but `container/list`, `container/heap` in the
  standard library, and the built-in `map` type IS a hash map
* **Rust**: a linked list node would typically be `Option<Box<Node>>` - Rust makes you be explicit about
  ownership/heap-allocation right in the type, something python's garbage collector hides from us entirely.
  `std::collections::BTreeMap` (a B-tree) and `HashMap` round out the standard library

## exercises

1. add a `reverse()` method to `LinkedList` that reverses the list *in place* (re-point every node's `.next`,
   don't just rebuild a new list) - O(n) time, O(1) extra space
2. add a `peek()` method to `Queue` that returns the front item WITHOUT removing it
3. add a `min_value()` method to `BinarySearchTree` that returns the smallest value in the tree (hint: in a
   BST the minimum is always the leftmost node - no need to check every value)
4. add a `__delitem__` method to `SimpleHashMap` so `del hm["some_key"]` works (find the right bucket, then
   remove that one `(key, value)` pair from it; raise `KeyError` if the key isn't there)

In [8]:
# TODO: exercise 1 - LinkedList.reverse()
# TODO: exercise 2 - Queue.peek()
# TODO: exercise 3 - BinarySearchTree.min_value()
# TODO: exercise 4 - SimpleHashMap.__delitem__

In [9]:
# sample solution 1
def _reverse(self):
    previous = None
    current = self.head
    while current is not None:
        current.next, previous, current = previous, current, current.next
    self.head = previous
LinkedList.reverse = _reverse

ll2 = LinkedList()
for v in (1, 2, 3, 4):
    ll2.append(v)
print("before reverse:", ll2)
ll2.reverse()
print("after reverse: ", ll2)

# sample solution 2
def _peek(self):
    return self._items[0]
Queue.peek = _peek

q2 = Queue()
q2.enqueue("first")
q2.enqueue("second")
print("\npeek (should not remove it):", q2.peek())
print("peek again, still there:", q2.peek())
print("now actually dequeue it:", q2.dequeue())

# sample solution 3
def _min_value(self):
    if self.root is None:
        raise ValueError("tree is empty")
    current = self.root
    while current.left is not None:
        current = current.left
    return current.value
BinarySearchTree.min_value = _min_value

print("\nsmallest value in our earlier bst:", bst.min_value())

# sample solution 4
def _delitem(self, key):
    bucket = self._bucket_for(key)
    for i, (existing_key, _) in enumerate(bucket):
        if existing_key == key:
            del bucket[i]
            self._count -= 1
            return
    raise KeyError(key)
SimpleHashMap.__delitem__ = _delitem

print("\nbefore delete, 'job' in map?", "job" in hm)
del hm["job"]
print("after delete, 'job' in map?", "job" in hm)

before reverse: 1 -> 2 -> 3 -> 4
after reverse:  4 -> 3 -> 2 -> 1

peek (should not remove it): first
peek again, still there: first
now actually dequeue it: first

smallest value in our earlier bst: 20

before delete, 'job' in map? True
after delete, 'job' in map? False


## what we learned

* how a singly linked list, stack, queue, binary search tree and hash map actually work underneath, by building
  simplified versions of each one ourselves
* WHY the real built-ins (`list`, `deque`, `dict`) are shaped the way they are, and picked concrete numbers
  (a timed `list.pop(0)` vs `deque.popleft()` race) to prove the difference isn't just theoretical
* that hash maps handle collisions via chaining, and grow (re-hash) themselves once they get too full - the
  same core idea `dict` uses internally
* in real code: reach for the standard library's version every time - the value of building these yourself is
  understanding, not replacing anything

next up: lesson 48 turns from data structures to *code quality* - clean code principles, SOLID, common
anti-patterns, and a few classic design patterns.